# Requêtes HTTP en Python avec `requests`

Dans ce notebook, vous allez apprendre à :

- comprendre les bases des **requêtes HTTP**
- utiliser la librairie **`requests`** (GET, POST, PUT, DELETE)
- envoyer et recevoir du **JSON**
- gérer les **headers**, **authentification**, **timeouts**
- manipuler des **API REST** (cas typiques en Data Science)
- télécharger des fichiers volumineux (streaming)
- traiter les **erreurs HTTP**

Ce module est essentiel dès que vous travaillez avec :
- des APIs web (OpenAI, HuggingFace, Kaggle…),
- des datastores REST,
- des microservices ML,
- des dashboards Streamlit/Flask qui communiquent avec un backend.

# 1. Introduction : qu’est-ce qu’une requête HTTP ?

Une requête HTTP est un message envoyé à un serveur pour demander une ressource.

Méthodes les plus courantes :

| Méthode | Rôle |
|---------|------|
| **GET** | Récupérer une ressource |
| **POST** | Envoyer des données (form, JSON) |
| **PUT** | Mettre à jour une ressource |
| **DELETE** | Supprimer une ressource |

Format courant d’échange : **JSON**  
Bibliothèque standard en Python : **`requests`**  

In [ ]:
import requests # A installer dans votre venv ! (avec pip, ou poetry, ou conda, ou uv)

# 2. Requête GET simple

On commence par récupérer une donnée depuis une API publique de test.

Exemple avec l’API "JSONPlaceholder".

In [2]:
url = "https://jsonplaceholder.typicode.com/posts/1"

response = requests.get(url)

In [3]:
print("Statut :", response.status_code)
print("Contenu brut :", response.text)
print("JSON :", response.json())

Statut : 200
Contenu brut : {
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}
JSON : {'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


# 3. Paramètres d’URL (`params`)

Pour ajouter des paramètres, on utilise le paramètre `params`.

Cas très courant en data science : pagination, filtre, limites, etc.

Le site `https://jsonplaceholder.typicode.com/` permet de tester nos requetes gratuitement

In [4]:
url = "https://jsonplaceholder.typicode.com/posts"

params = {
    "userId": 1
}

response = requests.get(url, params=params)

In [5]:
print("URL Finale :", response.url)
print("Résultat :", response.json()[:2])

URL Finale : https://jsonplaceholder.typicode.com/posts?userId=1
Résultat : [{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}, {'userId': 1, 'id': 2, 'title': 'qui est esse', 'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}]


# 4. Requête POST — envoyer des données

On envoie ici du JSON, par exemple un input envoyé à un microservice ML.

In [6]:
url = "https://jsonplaceholder.typicode.com/posts"

payload = {
    "title": "Mon modèle",
    "body": "Résultat de l'inférence",
    "userId": 123
}

response = requests.post(url, json=payload)

In [7]:
print("Statut :", response.status_code)
print("Réponse JSON :", response.json())

Statut : 201
Réponse JSON : {'title': 'Mon modèle', 'body': "Résultat de l'inférence", 'userId': 123, 'id': 101}


Note : Vous pouvez vous amuser a voir ce que les gens postent ;)

https://jsonplaceholder.typicode.com/posts

# 5. PUT — mettre à jour une ressource existante

In [8]:
url = "https://jsonplaceholder.typicode.com/posts/1"

payload = {
    "title": "Titre mis à jour",
    "body": "Nouveau contenu",
    "userId": 123
}

response = requests.put(url, json=payload)

In [9]:
print(response.json())

{'title': 'Titre mis à jour', 'body': 'Nouveau contenu', 'userId': 123, 'id': 1}


# 6. DELETE — supprimer une ressource

In [10]:
url = "https://jsonplaceholder.typicode.com/posts/1"

response = requests.delete(url)

In [11]:
print("Statut :", response.status_code)

Statut : 200


# 7. Headers — User-Agent, Token, etc.

Très utile pour :
- authentification,
- versionning API,
- simulation navigateur,
- API ML ou HuggingFace.

In [12]:
url = "https://jsonplaceholder.typicode.com/posts/1"

headers = {
    "User-Agent": "Cours-DataScience-Client",
    "Accept": "application/json"
}

In [13]:
response = requests.get(url, headers=headers)
print(response.json())

{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


# 8. Authentification

Beaucoup d’APIs nécessitent une clé :

- Bearer Token (OpenAI, HuggingFace…)
- API Key simple
- OAuth

Ici, exemple générique (clé fictive).

In [14]:
url = "https://api.exemple.com/data"

headers = {
    "Authorization": "Bearer VOTRE_TOKEN_ICI"
}

# 9. Timeout — éviter les blocages

Toujours utiliser un `timeout` en production.

In [15]:
try:
    response = requests.get("https://httpbin.org/delay/3", timeout=1)
except requests.exceptions.Timeout:
    print("⛔ Timeout atteint — le serveur met trop de temps à répondre.")

⛔ Timeout atteint — le serveur met trop de temps à répondre.


# 10. Gestion des erreurs HTTP

Bon réflexe pour les data pipelines → éviter les plantages silencieux.

In [16]:
response = requests.get("https://jsonplaceholder.typicode.com/does/not/exist")

if not response.ok:
    print("Erreur détectée :", response.status_code)
else:
    print(response.json())

Erreur détectée : 404


# 11. Télécharger un fichier (streaming)

Important pour les gros fichiers :

- datasets,
- modèles `.pt` / `.bin`,
- images téléchargées par lots.

Utiliser `stream=True` pour ne **pas tout charger en mémoire**.

In [17]:
url = "https://httpbin.org/image/png"

response = requests.get(url, stream=True)

with open("image.png", "wb") as f:
    for chunk in response.iter_content(chunk_size=2048):
        f.write(chunk)

print("Fichier téléchargé : image.png")

Fichier téléchargé : image.png


# 12. Cas Data Science : interroger un microservice ML

On simule un microservice de prédiction prenant un vecteur de features.

In [19]:
url = "https://jsonplaceholder.typicode.com/posts"

features = [0.2, 1.3, -0.7, 2.1]

payload = {
    "features": features,
    "model": "cnn_classifier_v1"
}

In [20]:
response = requests.post(url, json=payload)
print("Réponse :", response.json())

Réponse : {'features': [0.2, 1.3, -0.7, 2.1], 'model': 'cnn_classifier_v1', 'id': 101}


# Au final

Toutes ces notions sont indispensables pour :

- consommer des APIs ML (OpenAI, HuggingFace),
- construire des pipelines de données,
- communiquer entre microservices,
- créer des dashboards (Streamlit, Flask, FastAPI).